# BoxCommentator: LoRA fine-tune of Qwen2.5-1.5B

Trains the adapter on `finetune/data/train.jsonl` (generate it first with
`python -m finetune.gen_data --teacher nebius-teacher --max-usd 0.50` on your machine, then commit the files).

Notebook settings: Accelerator = **GPU T4**, Internet = **On**. Run training in this notebook, then start a
**separate session** for serving the adapter in vLLM (installing vLLM can replace torch).

In [ ]:
!nvidia-smi

In [ ]:
REPO_URL = "https://github.com/YOUR_USER/YOUR_REPO.git"   # <- set this
BRANCH = "main"
!git clone --depth 1 -b {BRANCH} {REPO_URL} repo
%cd repo
!pip install -q -r finetune/requirements.txt pyyaml openai

In [ ]:
import json, pathlib
for name in ["train", "eval"]:
    p = pathlib.Path(f"finetune/data/{name}.jsonl")
    assert p.exists(), f"{p} missing: run finetune/gen_data.py and commit finetune/data/"
    print(name, sum(1 for _ in open(p)), "examples")
print(json.dumps(json.load(open("finetune/data/manifest.json"))["train"], indent=1)[:400])

## Train (a few minutes on a T4)

In [ ]:
!python -m finetune.train_lora --data finetune/data --out /kaggle/working/boxer_lora

In [ ]:
import json, shutil
log = json.load(open("/kaggle/working/boxer_lora/train_log.json"))
print("eval loss:", log["eval_loss"])
shutil.make_archive("/kaggle/working/boxer_lora", "zip", "/kaggle/working/boxer_lora")
print("download boxer_lora.zip from the Output tab")

## Serve and benchmark the adapter (new session)
Upload `boxer_lora.zip` as a Kaggle dataset, attach it, install vLLM, then:

```
!pip install -q vllm pyyaml openai matplotlib
!python -m serving.run_matrix --plan lora --adapter /kaggle/input/boxer-lora --fights 10
!python -m bench.charts
```
The `lora` plan runs the base 1.5B and the adapter on identical fights so the comparison is fair.